<a href="https://colab.research.google.com/github/JustinStec/translations/blob/main/hebrew_english_translation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hebrew-to-English Book Translation

Translate a modern Hebrew book to English, chapter by chapter, with two translation backends:

| Backend | Quality | Speed | Cost |
|---------|---------|-------|------|
| **Helsinki-NLP MarianMT** | Good for straightforward prose | Fast (~1 page/sec on GPU) | Free |
| **Claude API** | Excellent for literary/nuanced text | ~1 page/min | API credits |

**Features:**
- Upload a Hebrew **PDF** -- text is extracted with Marker (GPU-accelerated OCR)
- Also accepts `.txt` files if you already have extracted text
- Automatic chapter detection and splitting
- Custom glossary for consistent terminology
- Side-by-side Hebrew/English display
- Export to `.txt`, `.docx`, or side-by-side HTML

**Before running:**
1. Go to **Runtime > Change runtime type > A100 GPU** (or T4 if unavailable)
2. If using Claude API: have your API key ready
3. Have your Hebrew book file (PDF or TXT) ready to upload

**New to Google Colab?** See the step-by-step guide: `INSTRUCTIONS.md` in this repository.

In [ ]:
# Cell 1: Install dependencies
!pip install -q marker-pdf             # GPU-accelerated PDF OCR
!pip install -q transformers sentencepiece torch
!pip install -q anthropic
!pip install -q python-docx
!pip install -q tqdm

In [ ]:
# Cell 2: Configuration
import os

# --- Translation backend ---
# Options: "marianmt" (free, GPU) or "claude" (API key required)
TRANSLATION_BACKEND = "marianmt"  # @param ["marianmt", "claude"]

# Claude API key (only needed if TRANSLATION_BACKEND = "claude")
CLAUDE_API_KEY = ""  # @param {type:"string"}

# Claude model to use for translation
CLAUDE_MODEL = "claude-sonnet-4-20250514"  # @param ["claude-sonnet-4-20250514", "claude-haiku-4-5-20251001"]

# --- Chunking ---
# Max characters per translation chunk (MarianMT works best with shorter segments)
MARIANMT_CHUNK_CHARS = 400
# Claude can handle much larger chunks for better context
CLAUDE_CHUNK_CHARS = 6000

# --- Directories ---
UPLOAD_DIR = "/content/hebrew_source"
OUTPUT_DIR = "/content/translation_output"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set CUDA for marker
os.environ["TORCH_DEVICE"] = "cuda"

print(f"Backend: {TRANSLATION_BACKEND}")
print(f"Output directory: {OUTPUT_DIR}")
if TRANSLATION_BACKEND == "claude" and not CLAUDE_API_KEY:
    print("\nWARNING: Set CLAUDE_API_KEY above before running translation cells.")

In [ ]:
# Cell 3: Initialize Marker (GPU-accelerated OCR)
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

print("\nLoading Marker models (this takes 1-2 minutes)...")

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

model_dict = create_model_dict()
converter = PdfConverter(artifact_dict=model_dict)

print("Marker models loaded and ready.")

In [ ]:
# Cell 4: Upload and OCR your Hebrew book
from google.colab import files as colab_files
import glob
import re
import unicodedata
import gc

# --- Text cleaning functions (same as reocr_marker_colab.ipynb) ---

def clean_text(text):
    """Thorough UTF-8 cleaning for OCR output."""
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)

    ligature_map = {
        "\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl",
        "\ufb03": "ffi", "\ufb04": "ffl", "\ufb05": "st", "\ufb06": "st",
    }
    for lig, replacement in ligature_map.items():
        text = text.replace(lig, replacement)

    quote_map = {
        "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
        "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"',
        "\u2032": "'", "\u2033": '"',
        "\u2013": "-", "\u2014": "--", "\u2015": "--",
        "\u2026": "...", "\u00a0": " ",
        "\u200b": "", "\u200c": "", "\u200d": "", "\ufeff": "", "\u00ad": "",
    }
    for char, replacement in quote_map.items():
        text = text.replace(char, replacement)

    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'[\ud800-\udfff]', '', text)
    text = re.sub(r'\\u[0-9a-fA-F]{0,3}[^0-9a-fA-F]', '', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    text = re.sub(r'\n\s*\d{1,4}\s*\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = text.strip()

    return text


def strip_markdown_for_txt(md_text):
    """Convert Marker's markdown output to clean plain text."""
    text = md_text
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', text)
    text = re.sub(r'_{1,3}([^_]+)_{1,3}', r'\1', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'!\[([^\]]*)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'^[-*_]{3,}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'```[^`]*```', '', text, flags=re.DOTALL)
    text = re.sub(r'`([^`]+)`', r'\1', text)
    return text.strip()


# --- Upload ---
print("Upload your Hebrew book (PDF or TXT):")
print("  - PDF: will be OCR'd with Marker (GPU-accelerated, best quality)")
print("  - TXT: must be UTF-8 encoded")
print()

uploaded = colab_files.upload()

source_files = []

for filename, content in uploaded.items():
    filepath = os.path.join(UPLOAD_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(content)
    print(f"\nUploaded: {filename} ({len(content):,} bytes)")

    if filename.lower().endswith(".pdf"):
        print(f"  Running Marker OCR (this may take a few minutes)...")

        result = converter(filepath)
        md_text = result.markdown

        if not md_text or len(md_text.strip()) < 50:
            print(f"  WARNING: Marker extracted very little text.")
            print(f"  The PDF may be image-only or corrupted.")
            continue

        # Clean: markdown -> plain text -> UTF-8 cleanup
        plain_text = strip_markdown_for_txt(md_text)
        cleaned = clean_text(plain_text)
        word_count = len(cleaned.split())

        # Save extracted text
        txt_filename = filename.rsplit(".", 1)[0] + ".txt"
        txt_path = os.path.join(UPLOAD_DIR, txt_filename)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(cleaned)

        print(f"  Extracted {word_count:,} words")
        print(f"  Saved as: {txt_filename}")

        # Preview
        preview = cleaned[:500]
        print(f"\n  --- Preview of extracted text ---")
        print(f"  {preview}")
        print(f"  --- End preview ---\n")

        source_files.append(txt_path)

    elif filename.lower().endswith(".txt"):
        source_files.append(filepath)
    else:
        print(f"  WARNING: Unsupported file type. Use .pdf or .txt files.")

source_files.sort()
print(f"\nTotal source files ready: {len(source_files)}")
for f in source_files:
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f)} ({size:,} bytes)")

In [ ]:
# Cell 5: Free Marker GPU memory, then load translation model
import gc

# Unload Marker to free GPU memory for the translation model
print("Unloading Marker models to free GPU memory...")
del converter
del model_dict
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.\n")

# --- Load translation backend ---
translator_model = None
translator_tokenizer = None
claude_client = None

device = "cuda" if torch.cuda.is_available() else "cpu"

if TRANSLATION_BACKEND == "marianmt":
    from transformers import MarianMTModel, MarianTokenizer

    MODEL_NAME = "Helsinki-NLP/opus-mt-tc-big-he-en"
    print(f"Loading {MODEL_NAME}...")

    translator_tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
    translator_model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
    translator_model.eval()

    print(f"MarianMT model loaded on {device}.")

elif TRANSLATION_BACKEND == "claude":
    import anthropic

    if not CLAUDE_API_KEY:
        raise ValueError("Set CLAUDE_API_KEY in Cell 2 before running this cell.")

    claude_client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    print(f"Claude client initialized (model: {CLAUDE_MODEL}).")

In [ ]:
# Cell 6: Translation functions + glossary
import time

# --- Custom glossary ---
# Add Hebrew terms and their preferred English translations here.
# These will be applied as post-processing replacements for MarianMT,
# or included in the prompt for Claude.
GLOSSARY = {
    # Hebrew term -> preferred English translation
    # Example entries (uncomment and modify as needed):
    # "\u05e9\u05d9\u05e8\u05d4": "poetry",
    # "\u05e1\u05d5\u05e4\u05e8": "author",
    # "\u05e4\u05e8\u05e7": "chapter",
}


def split_into_sentences(text):
    """
    Split Hebrew text into sentences.
    Handles Hebrew punctuation patterns including sof pasuq.
    """
    sentences = re.split(r'(?<=[.!?\u05C3])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


def chunk_text(text, max_chars):
    """
    Split text into chunks at sentence boundaries, respecting max_chars.
    """
    sentences = split_into_sentences(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sentence in sentences:
        sent_len = len(sentence)

        if sent_len > max_chars:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
                current_chunk = []
                current_len = 0
            chunks.append(sentence)
            continue

        if current_len + sent_len + 1 > max_chars:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_len = sent_len
        else:
            current_chunk.append(sentence)
            current_len += sent_len + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def translate_marianmt(text):
    """
    Translate Hebrew text to English using MarianMT.
    Handles chunking internally for long texts.
    """
    chunks = chunk_text(text, MARIANMT_CHUNK_CHARS)
    translations = []

    for chunk in chunks:
        inputs = translator_tokenizer(
            chunk, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = translator_model.generate(
                **inputs,
                num_beams=5,
                max_length=512,
                early_stopping=True,
            )

        translated = translator_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(translated)

    result = " ".join(translations)

    # Apply glossary replacements
    for hebrew_term, english_term in GLOSSARY.items():
        result = result.replace(hebrew_term, english_term)

    return result


def translate_claude(text, chapter_title="", context=""):
    """
    Translate Hebrew text to English using Claude.
    Provides glossary and context for higher-quality literary translation.
    """
    glossary_str = ""
    if GLOSSARY:
        glossary_lines = [f"  {heb} -> {eng}" for heb, eng in GLOSSARY.items()]
        glossary_str = (
            "\n\nUse these preferred translations for specific terms:\n"
            + "\n".join(glossary_lines)
        )

    context_str = ""
    if context:
        context_str = f"\n\nFor context, here is the preceding passage's translation:\n{context[-1500:]}"

    chapter_str = ""
    if chapter_title:
        chapter_str = f" from the chapter titled '{chapter_title}'"

    system_prompt = (
        "You are an expert literary translator specializing in modern Hebrew to English translation. "
        "Produce natural, fluent English that preserves the author's voice, tone, and style. "
        "Maintain paragraph structure. Transliterate proper nouns unless they have standard English forms. "
        "Do not add commentary or notes -- output only the English translation."
        f"{glossary_str}"
    )

    user_prompt = (
        f"Translate the following modern Hebrew passage{chapter_str} into English."
        f"{context_str}"
        f"\n\n---\n\n{text}"
    )

    response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )

    return response.content[0].text


def translate(text, chapter_title="", context=""):
    """Route to the configured translation backend."""
    if TRANSLATION_BACKEND == "marianmt":
        return translate_marianmt(text)
    elif TRANSLATION_BACKEND == "claude":
        return translate_claude(text, chapter_title, context)
    else:
        raise ValueError(f"Unknown backend: {TRANSLATION_BACKEND}")


# Quick test
test_hebrew = "\u05e9\u05dc\u05d5\u05dd \u05e2\u05d5\u05dc\u05dd. \u05d6\u05d4 \u05de\u05e9\u05e4\u05d8 \u05e4\u05e9\u05d5\u05d8 \u05dc\u05d1\u05d3\u05d9\u05e7\u05d4."
test_result = translate(test_hebrew)
print(f"Test input:  {test_hebrew}")
print(f"Test output: {test_result}")

In [ ]:
# Cell 7: Detect and split chapters
import json

# Common Hebrew chapter heading patterns
CHAPTER_PATTERNS = [
    r'^\u05e4\u05e8\u05e7\s+[\u05d0-\u05ea\d]+',          # פרק + number/letter
    r'^\u05e4\u05e8\u05e7\s+.{1,60}$',                      # פרק + title
    r'^\u05d7\u05dc\u05e7\s+[\u05d0-\u05ea\d]+',          # חלק + number/letter
    r'^\u05e9\u05e2\u05e8\s+[\u05d0-\u05ea\d]+',          # שער + number/letter
    r'^\d+\.\s+.{1,60}$',                                    # Numbered: "1. Title"
    r'^[A-Z\u05d0-\u05ea]{1,3}[.)]\s',                      # Letter numbering: A. or א.
]


def detect_chapters(text):
    """
    Detect chapter breaks in Hebrew text.
    Returns list of (title, content) tuples.
    """
    lines = text.split("\n")
    chapter_indices = []

    combined_pattern = "|".join(f"({p})" for p in CHAPTER_PATTERNS)

    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if len(stripped) < 80 and re.match(combined_pattern, stripped):
            chapter_indices.append(i)

    if not chapter_indices:
        return [("Full Text", text)]

    chapters = []
    for idx, start in enumerate(chapter_indices):
        title = lines[start].strip()
        end = chapter_indices[idx + 1] if idx + 1 < len(chapter_indices) else len(lines)
        content = "\n".join(lines[start + 1:end]).strip()
        if content:
            chapters.append((title, content))

    if chapter_indices[0] > 0:
        preamble = "\n".join(lines[:chapter_indices[0]]).strip()
        if preamble:
            chapters.insert(0, ("Preamble", preamble))

    return chapters


# --- Load and split all source files ---
all_chapters = []

for filepath in source_files:
    filename = os.path.basename(filepath)
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    chapters = detect_chapters(text)
    print(f"\n{filename}: {len(chapters)} chapter(s) detected")
    for title, content in chapters:
        word_count = len(content.split())
        print(f"  {title} ({word_count:,} words)")
        all_chapters.append({
            "source_file": filename,
            "title": title,
            "hebrew": content,
        })

print(f"\nTotal chapters to translate: {len(all_chapters)}")
total_words = sum(len(ch["hebrew"].split()) for ch in all_chapters)
print(f"Total words: {total_words:,}")

In [ ]:
# Cell 8: Translate all chapters
from tqdm import tqdm

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "translation_checkpoint.json")

# Load checkpoint if resuming
completed_translations = {}
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        completed_translations = json.load(f)
    print(f"Resuming from checkpoint: {len(completed_translations)} chapters already translated")

max_chars = CLAUDE_CHUNK_CHARS if TRANSLATION_BACKEND == "claude" else MARIANMT_CHUNK_CHARS
translation_errors = []

print(f"\nTranslating {len(all_chapters)} chapters using {TRANSLATION_BACKEND}...")
print(f"Chunk size: {max_chars} chars\n")

for i, chapter in enumerate(tqdm(all_chapters)):
    chapter_key = f"{chapter['source_file']}::{chapter['title']}"

    if chapter_key in completed_translations:
        chapter["english"] = completed_translations[chapter_key]
        continue

    hebrew_text = chapter["hebrew"]

    try:
        if TRANSLATION_BACKEND == "claude":
            chunks = chunk_text(hebrew_text, max_chars)
            translated_chunks = []
            context = ""

            for chunk in chunks:
                translated = translate_claude(chunk, chapter["title"], context)
                translated_chunks.append(translated)
                context = translated
                time.sleep(1)

            chapter["english"] = "\n\n".join(translated_chunks)
        else:
            paragraphs = hebrew_text.split("\n\n")
            translated_paragraphs = []

            for para in paragraphs:
                para = para.strip()
                if not para:
                    continue
                translated_paragraphs.append(translate_marianmt(para))

            chapter["english"] = "\n\n".join(translated_paragraphs)

        completed_translations[chapter_key] = chapter["english"]

        word_count = len(chapter["english"].split())
        tqdm.write(f"  {chapter['title']}: {word_count:,} words translated")

    except Exception as e:
        translation_errors.append({"chapter": chapter_key, "error": str(e)})
        tqdm.write(f"  ERROR on {chapter['title']}: {e}")
        chapter["english"] = f"[TRANSLATION ERROR: {e}]"

    # Checkpoint every 5 chapters
    if (i + 1) % 5 == 0:
        with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            json.dump(completed_translations, f, ensure_ascii=False, indent=2)
        if TRANSLATION_BACKEND == "marianmt":
            torch.cuda.empty_cache()

# Final checkpoint
with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
    json.dump(completed_translations, f, ensure_ascii=False, indent=2)

print(f"\nTranslation complete!")
print(f"Errors: {len(translation_errors)}")
if translation_errors:
    for err in translation_errors:
        print(f"  - {err['chapter']}: {err['error']}")

In [ ]:
# Cell 9: Side-by-side preview
from IPython.display import HTML, display

def preview_chapter(chapter_index=0):
    """Display a side-by-side preview of a translated chapter."""
    ch = all_chapters[chapter_index]
    hebrew = ch["hebrew"][:3000]
    english = ch.get("english", "[Not yet translated]")[:3000]

    hebrew_html = hebrew.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
    english_html = english.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")

    html = f"""
    <h3>{ch['title']} (from {ch['source_file']})</h3>
    <div style="display: flex; gap: 20px;">
      <div style="flex: 1; direction: rtl; font-size: 14px; line-height: 1.8;
                  border: 1px solid #ccc; padding: 15px; border-radius: 8px;
                  background: #fafafa; max-height: 600px; overflow-y: auto;">
        <h4 style="direction: ltr;">Hebrew (original)</h4>
        {hebrew_html}
      </div>
      <div style="flex: 1; font-size: 14px; line-height: 1.8;
                  border: 1px solid #ccc; padding: 15px; border-radius: 8px;
                  background: #fafafa; max-height: 600px; overflow-y: auto;">
        <h4>English (translation)</h4>
        {english_html}
      </div>
    </div>
    """
    display(HTML(html))


# Show preview of first chapter
if all_chapters:
    print(f"Chapters available: {len(all_chapters)}")
    print("Change the index below to preview a different chapter.\n")
    preview_chapter(0)

In [ ]:
# Cell 10: Export translations
from google.colab import files as colab_files

def export_plain_text():
    """Export English translation as a single .txt file."""
    out_path = os.path.join(OUTPUT_DIR, "translation_english.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        for ch in all_chapters:
            f.write(f"{'=' * 60}\n")
            f.write(f"{ch['title']}\n")
            f.write(f"{'=' * 60}\n\n")
            f.write(ch.get("english", "[Not translated]"))
            f.write("\n\n\n")
    print(f"Saved: {out_path}")
    return out_path


def export_bilingual_text():
    """Export side-by-side Hebrew/English as a .txt file."""
    out_path = os.path.join(OUTPUT_DIR, "translation_bilingual.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        for ch in all_chapters:
            f.write(f"{'=' * 60}\n")
            f.write(f"{ch['title']}\n")
            f.write(f"{'=' * 60}\n\n")
            f.write("--- HEBREW ---\n\n")
            f.write(ch["hebrew"])
            f.write("\n\n--- ENGLISH ---\n\n")
            f.write(ch.get("english", "[Not translated]"))
            f.write("\n\n\n")
    print(f"Saved: {out_path}")
    return out_path


def export_docx():
    """Export English translation as a .docx file."""
    from docx import Document
    from docx.shared import Pt, Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH

    doc = Document()

    title_para = doc.add_paragraph()
    title_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title_para.add_run("Translation from Hebrew")
    run.font.size = Pt(24)
    run.bold = True

    doc.add_paragraph()

    source_para = doc.add_paragraph()
    source_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = source_para.add_run(f"Translated using {TRANSLATION_BACKEND}")
    run.font.size = Pt(12)
    run.italic = True

    doc.add_page_break()

    for ch in all_chapters:
        doc.add_heading(ch["title"], level=1)
        english = ch.get("english", "[Not translated]")
        for para_text in english.split("\n\n"):
            para_text = para_text.strip()
            if para_text:
                p = doc.add_paragraph(para_text)
                p.paragraph_format.space_after = Pt(6)
                for run in p.runs:
                    run.font.size = Pt(11)
        doc.add_page_break()

    out_path = os.path.join(OUTPUT_DIR, "translation_english.docx")
    doc.save(out_path)
    print(f"Saved: {out_path}")
    return out_path


def export_bilingual_html():
    """Export side-by-side HTML for easy reading."""
    out_path = os.path.join(OUTPUT_DIR, "translation_bilingual.html")

    html_parts = ["""
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Hebrew-English Translation</title>
  <style>
    body { font-family: Georgia, serif; margin: 40px; background: #f9f9f9; }
    h1 { text-align: center; color: #333; }
    .chapter { margin: 40px 0; }
    .chapter h2 { color: #555; border-bottom: 2px solid #ddd; padding-bottom: 8px; }
    .columns { display: flex; gap: 30px; }
    .col-he { flex: 1; direction: rtl; text-align: right; font-size: 15px;
              line-height: 2; padding: 20px; background: #fff;
              border: 1px solid #ddd; border-radius: 8px; }
    .col-en { flex: 1; font-size: 15px; line-height: 1.8;
              padding: 20px; background: #fff;
              border: 1px solid #ddd; border-radius: 8px; }
    .col-label { font-weight: bold; color: #888; margin-bottom: 10px;
                 direction: ltr; text-align: left; }
  </style>
</head>
<body>
<h1>Hebrew-English Translation</h1>
"""]

    for ch in all_chapters:
        heb = ch["hebrew"].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        eng = ch.get("english", "[Not translated]").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")

        html_parts.append(f"""
<div class="chapter">
  <h2>{ch['title']}</h2>
  <div class="columns">
    <div class="col-he">
      <div class="col-label">Hebrew</div>
      {heb}
    </div>
    <div class="col-en">
      <div class="col-label">English</div>
      {eng}
    </div>
  </div>
</div>
""")

    html_parts.append("</body></html>")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(html_parts))

    print(f"Saved: {out_path}")
    return out_path


# --- Run exports ---
print("Exporting translations...\n")

txt_path = export_plain_text()
bilingual_path = export_bilingual_text()
docx_path = export_docx()
html_path = export_bilingual_html()

print("\nDownloading files...")
colab_files.download(txt_path)
colab_files.download(docx_path)
colab_files.download(html_path)

## Done!

Your translation files have been exported. Here's what you got:

| File | Description |
|------|-------------|
| `translation_english.txt` | Plain English translation |
| `translation_english.docx` | Formatted Word document |
| `translation_bilingual.txt` | Hebrew + English interleaved |
| `translation_bilingual.html` | Side-by-side HTML for browser reading |

**Tips for better results:**
- **MarianMT** works well for straightforward modern Hebrew prose. If you see odd output, try adding terms to the `GLOSSARY` in Cell 6.
- **Claude** produces much higher quality literary translations. Use `claude-sonnet-4-20250514` for the best balance of quality and cost.
- For a full book, consider translating one chapter first with both backends to compare quality before committing to the full run.
- The checkpoint system saves progress automatically -- if the runtime disconnects, just re-run and it will resume where it left off.